In [1]:
from pathlib import Path

from stable_platform_matchings import InstanceGenerator, Optimizer, OptimizerParams, SolverOptions

In [2]:
SIM_SIZE = 12
N_INTS = 12
SEED = 67

INDO_CRS = "EPSG:23867"
DATA_DIR = Path("../../SyntheticInstanceGenerator/data/")

FARMERS_PATH = DATA_DIR / "farmers.csv"
FARMERS_14_PATH = DATA_DIR / "farmers_14.csv"
INTS_PATH = DATA_DIR / "intermediaries.csv"
GRAPH_PATH = DATA_DIR / "graph_0-14960_00_new.pickle"
ALPHA_PATH = DATA_DIR / "precomputed_alpha.json"
SIGMAS_PATH = DATA_DIR / "precomputed_sigmas.json"

In [3]:
ig = InstanceGenerator(
    FARMERS_PATH, FARMERS_14_PATH, INTS_PATH, GRAPH_PATH, ALPHA_PATH, SIGMAS_PATH
)

In [4]:
ig.gen_intermediaries(n_intermediaries=1, seed=SEED)
ig.gen_calendar(seed=SEED, scale=1, n_cycles=2)

In [5]:
ig.calendar_df

,scaled_quantity,day,farmer_lat,farmer_lon,intermediary_id,farmer_id
1,1.1,1,-0.699271,102.459096,nostalgic_bohr,nostalgic_bohr_d0_f0
2,1.1,19,-0.699271,102.459096,nostalgic_bohr,nostalgic_bohr_d0_f0
3,1.1,24,-0.699271,102.459096,nostalgic_bohr,nostalgic_bohr_d0_f0
5,1.1,2,-0.681234,102.425442,nostalgic_bohr,nostalgic_bohr_d1_f0
6,1.1,15,-0.681234,102.425442,nostalgic_bohr,nostalgic_bohr_d1_f0
...,...,...,...,...,...,...
113,1.0,16,-0.708314,102.443404,nostalgic_bohr,nostalgic_bohr_d13_f1
114,1.0,25,-0.708314,102.443404,nostalgic_bohr,nostalgic_bohr_d13_f1
117,0.8,14,-0.721851,102.454628,nostalgic_bohr,nostalgic_bohr_d13_f2
118,0.8,27,-0.721851,102.454628,nostalgic_bohr,nostalgic_bohr_d13_f2


In [6]:
platform = ig.gen_instance(instance_id="hello", day=15, n_hist_sets=1)

In [7]:
epsilons = {intermediary.id: 2 for intermediary in platform.intermediaries}
het_costs = {
    intermediary.id: (platform.dist_to_mill[intermediary.id] * 2)
    for intermediary in platform.intermediaries
}

In [8]:
params = OptimizerParams(
    het_costs=het_costs,
    epsilons=epsilons,
    backend="gurobi",
    vrp_mode="approximate",
    verbose=True,
    print_width=80,
)

opt = Optimizer(platform, params)



============================= Optimizer Parameters =============================
---------------------------------- het_costs -----------------------------------
  {'nostalgic_bohr': 229550.8889446209}
----------------------------------- epsilons -----------------------------------
  {'nostalgic_bohr': 2}
  backend                    gurobi
  vrp_mode                   approximate
  verbose                    True
  print_width                80


============================== VRP Initialization ==============================
---------------------- Solving for minimum cost matching -----------------------
Set parameter Username
Set parameter LicenseID to value 2717834
Academic license - for non-commercial use only - expires 2026-10-03
Set parameter Threads to value 0
Set parameter TimeLimit to value 2000
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.3.0 25D2128)

CPU model: Apple M4 Pro
Thread count: 14 physical cores, 14 logical processors, using up to 14

In [9]:
options = SolverOptions(
    strategy="heuristic_optimized",
    structured_farmer_payments=False,
    dominance_constraints=False,
    pay_unmatched=False,
    aggregate=True,
)


solution = opt.solve(options=options)

if solution.max_intermediary_welfare_solution is not None:
    print(solution.max_intermediary_welfare_solution.platform_profit / platform.lc_to_usd)



================================ Solver Options ================================
  Strategy                   heuristic_optimized
  Structured Farmer Payments False
  Dominance Constraints      False
  Early Stop                 False
  Aggregate                  True
  Pay Unmatched              False


======================== Strategy: heuristic_optimized =========================
  Farmers                    5
  Intermediaries             1


============================== Branch Evaluation ===============================
  Forced matched:
    []
  Forced unmatched:
    []

Set parameter Threads to value 0
----------------------- Primal Solve: Forced Lower Bound -----------------------
  Platform-profit objective  -0.000
  Intermediary-welfare objective 13,625,639.026
  Farmer-welfare objective   13,625,639.026
  New rows                   5
  Intermediary probabilities:
    {'nostalgic_bohr': 1.0}
---------------------------- Lower-Bound Candidate -----------------------------
 